## Clinical Information Extraction with LLMs

This notebook demonstrates a pipeline for extracting structured clinical information
from emergency room (ER) reports using a large language model. It extracts:
- precipitating event (cause of ER visit)
- confidence
- explanation

### Pipeline

1. Clinical text extraction
2. Prompt-based information extraction
3. Structured JSON parsing

In [1]:
import json

from src.data_loader.preprocessing import ClinicalPreprocessor
from src.information_extraction.extractor import MedicalExtractor
from src.information_extraction.prompts import build_chat_prompt


In [2]:
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

extractor = MedicalExtractor(MODEL_NAME)


Loading model: mistralai/Mistral-7B-Instruct-v0.3...


/home/tatiana.bladier/.local/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


In [3]:
en_markers = ["CHIEF COMPLAINT:", 
            "HISTORY OF PRESENT ILLNESS:", 
            "REASON FOR VISIT:"]

en_stop_marker = "About This Sample:"

en_preprocessor = ClinicalPreprocessor(
    language="en", 
    markers=en_markers, 
    end_marker=en_stop_marker
)

In [4]:
# ── 2. Prompt ─────────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are a medical information extraction assistant.
Given a medical text, extract structured information and return it as JSON only,
with no preamble or explanation.
"""

USER_INSTRUCTION_TEMPLATE = """Identify the precipitating event that led to the medical emergency described in the text.

A precipitating event is a concrete external incident such as: a fall, a car accident, 
domestic violence, a suicide attempt, a sports injury, an overdose, a workplace accident, etc.

Do NOT return a diagnosis or symptom as the event (e.g. "myocardial infarction", "chest pain" are NOT events).
If no such external incident is mentioned, return null.

Example of valid output when event is known:
{{"event": "fall from ladder", "confidence": "high", "explanation": "The patient fell from a ladder."}}

Example of valid output when event is unknown:
{{"event": null, "confidence": "unknown", "explanation": "No precipitating event is clearly stated in the text."}}

Return ONLY a valid JSON object with exactly these fields:
- "event": the precipitating external incident in a few words, or null if not mentioned
- "confidence": "high" / "medium" / "low" / "unknown"
- "explanation": one sentence justifying your answer

Medical text:
{text}
"""

In [5]:
# Prompt example

def build_prompt(text: str) -> list:
    """
    Constructs a structured chat-style prompt for extracting medical events.
    Returns a list of messages (system and user) for the model.
    """
    user_content = USER_INSTRUCTION_TEMPLATE.format(text=text)
    return [
        {
            "role": "system", 
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user", 
            "content": user_content
        }
    ]

In [6]:
en_triage_notes_raw = "data/raw_data/mtsamples_er/texts"

df_en = en_preprocessor.process_folder(
    en_triage_notes_raw, 
    extractor_instance=extractor, 
    prompt_func=lambda x: build_chat_prompt(x, task="cause"),
    limit=10
)
df_en

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


,event,confidence,explanation,source_file,lang
0,None,unknown,No precipitating event is clearly stated in th...,Abdominal_Pain_-_Consult.txt,en
1,None,unknown,No precipitating event is clearly stated in th...,Abdominal_Pain_2D_Consult.txt,en
2,fall off bicycle,high,The patient was admitted to the Emergency Depa...,Abrasions_26_Lacerations_2D_ER_Visit.txt,en
3,fall off bicycle,high,The patient was admitted to the Emergency Depa...,Abrasions__Lacerations_-_ER_Visit.txt,en
4,accidental ingestion of Celesta,high,The patient ingested two to three tablets of C...,Accidental_Celesta_Ingestion_-_ER_Visit.txt,en
5,accidental ingestion of Celesta,high,The patient ingested two to three tablets of C...,Accidental_Celesta_Ingestion_2D_ER_Visit.txt,en
6,None,unknown,No precipitating event is clearly stated in th...,Acute_Inferior_Myocardial_Infarction.txt,en
7,None,unknown,The text does not clearly state a precipitatin...,Agitation_-_ER_Visit.txt,en
8,None,unknown,The text does not clearly state a precipitatin...,Agitation_2D_ER_Visit.txt,en
9,"Unclear, possibly related to alcohol abuse or ...",medium,The text mentions the patient's history of alc...,Air_Under_Diaphragm_-_Consult.txt,en


In [7]:
df_en["event"].value_counts()

event
fall off bicycle                                          2
accidental ingestion of Celesta                           2
Unclear, possibly related to alcohol abuse or sedation    1
Name: count, dtype: int64

In [8]:
for _, row in df_en.iterrows():
    print(f"--- {row['source_file']} ---")
    print(json.dumps(row.to_dict(), indent=4))

--- Abdominal_Pain_-_Consult.txt ---
{
    "event": null,
    "confidence": "unknown",
    "explanation": "No precipitating event is clearly stated in the text.",
    "source_file": "Abdominal_Pain_-_Consult.txt",
    "lang": "en"
}
--- Abdominal_Pain_2D_Consult.txt ---
{
    "event": null,
    "confidence": "unknown",
    "explanation": "No precipitating event is clearly stated in the text.",
    "source_file": "Abdominal_Pain_2D_Consult.txt",
    "lang": "en"
}
--- Abrasions_26_Lacerations_2D_ER_Visit.txt ---
{
    "event": "fall off bicycle",
    "confidence": "high",
    "explanation": "The patient was admitted to the Emergency Department due to a fall off his bicycle, which is a concrete external incident.",
    "source_file": "Abrasions_26_Lacerations_2D_ER_Visit.txt",
    "lang": "en"
}
--- Abrasions__Lacerations_-_ER_Visit.txt ---
{
    "event": "fall off bicycle",
    "confidence": "high",
    "explanation": "The patient was admitted to the Emergency Department due to a fall o

### Evaluation